<a href="https://colab.research.google.com/github/jnolorbe/hydrology-peru/blob/main/notebooks/05_analisis_por_eventos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05. ENSO Event Analysis (formerly *script* 09)

Response of terrestrial water storage (TWS, GRACE/GRACE-FO JPL RL06.3Mv04) to five events with known impacts in
Peru, including those that the regime classification does not assign: El Niño 2009–2010, La Niña 2010–2012,
El Niño 2015–2016 and the coastal events of 2017 and 2023–2024 (thesis, Sections 3.6.9 and 4.4.4).

**Windows.** They are defined with the indices and not with the TWS, so that the choice of the extremes does not
determine the sign of the change: a continuous block with ONI ≥ +0.5 (2009–2010 and 2015–2016), ONI ≤ −0.5
(La Niña 2010–2012; gaps of up to three months are merged) and ICEN ≥ +0.4 (coastal events, ENFEN criterion).

**Metric.** With the deseasonalized series of each domain (as in Notebook 02),
ΔS = mean of the last three months of the window − mean of the three months before its start, in cm and in Gt
(1 cm over 1 km² = 10⁻⁵ Gt). As a reference, the percentile of ΔS among all windows of the same length in
2002–2025 is given: it is descriptive, not a hypothesis test, because the windows overlap and the series is
autocorrelated. The analysis is repeated without the linear trend and with windows defined by the E and C indices.


# 1. Environment and Data


In [1]:
!pip install --quiet netCDF4 xarray rioxarray geopandas shapely statsmodels

import os
import numpy as np
import pandas as pd

REPO_NAME = "hydrology-peru"
GITHUB_USERNAME = "jnolorbe"  # change if you use your own copy
REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
else:
    %cd {REPO_NAME}
    !git pull
    %cd ..

/home/claude/tr/run/hydrology-peru
/home/claude/tr/run


In [2]:
# ── Shared repository functions (notebooks/comun.py) ───────────────────────────
import sys, os, importlib
sys.path.insert(0, os.path.join(REPO_NAME, 'notebooks'))
import comun; importlib.reload(comun)
from comun import *
DERIV = os.path.join(REPO_NAME, 'data', 'processed_derived')
os.makedirs(DERIV, exist_ok=True)
pd.set_option('display.width', 200, 'display.max_columns', 30)
verificar(REPO_NAME)

Verificación correcta: 288 meses, 14 ámbitos, El Niño E 13 meses en 5 episodios, 64 registros in situ.


# 2. Event Windows and Storage Change


In [3]:
ev = analisis_eventos(REPO_NAME)
ev['ventanas'].to_csv(os.path.join(DERIV, 'event_windows.csv'), index=False)
ev['cambios'].to_csv(os.path.join(DERIV, 'event_change.csv'), index=False)
ev['sensibilidad'].to_csv(os.path.join(DERIV, 'event_sensitivity_EC.csv'), index=False)
display(ev['ventanas'].round(2))

,evento,inicio,fin,meses,imputados,E_max,E_min,C_max,C_min
0,El Niño 2009–10,2009-08,2010-03,8,0,0.67,-0.10,1.82,0.53
1,La Niña 2010–12,2010-06,2012-02,21,3,0.84,-0.76,-0.56,-2.19
2,El Niño 2015–16,2014-10,2016-04,19,6,2.40,-0.96,1.96,0.61
3,El Niño costero 2017,2016-12,2017-04,5,1,0.94,-0.12,-0.01,-0.41
4,El Niño costero 2023–24,2023-02,2024-02,13,0,2.53,0.39,2.00,-1.11


In [4]:
cambios = ev['cambios']
tabla_cm = cambios[cambios.variante == 'base'].pivot(index='ambito', columns='evento', values='dS_cm').loc[AMBITOS, ev['eventos']]
tabla_pc = cambios[cambios.variante == 'base'].pivot(index='ambito', columns='evento', values='percentil').loc[AMBITOS, ev['eventos']]
print('ΔS by domain (cm)'); display(tabla_cm.rename(index=NOMBRE_ES).round(1))
print('Percentile among windows of the same length'); display(tabla_pc.rename(index=NOMBRE_ES).round(0))
print('Total ΔS of the 14 domains (Gt)')
display(cambios.groupby(['variante', 'evento'], sort=False).dS_Gt.sum().round(1).unstack()[ev['eventos']])

ΔS by domain (cm)


evento,El Niño 2009–10,La Niña 2010–12,El Niño 2015–16,El Niño costero 2017,El Niño costero 2023–24
ambito,,,,,
Jequetepeque-Zarumilla,-0.4,0.2,1.4,2.2,4.0
Huarmey-Chicama,-7.9,2.1,-0.7,0.2,4.5
Marañón,-7.4,3.4,-1.2,2.9,5.7
Huallaga,-10.0,5.5,-4.0,4.8,5.7
Amazonas,-10.7,10.2,-6.5,5.6,8.0
Cañete-Fortaleza,-1.3,1.9,-1.1,-0.7,1.6
Mantaro,-2.5,4.6,-2.3,-1.5,5.0
Cháparra-Chincha,-0.7,4.7,-2.1,-1.2,1.4
Ucayali,-8.6,8.7,-8.3,-0.8,7.1


Percentile among windows of the same length


evento,El Niño 2009–10,La Niña 2010–12,El Niño 2015–16,El Niño costero 2017,El Niño costero 2023–24
ambito,,,,,
Jequetepeque-Zarumilla,34.0,52.0,79.0,97.0,100.0
Huarmey-Chicama,0.0,73.0,35.0,53.0,96.0
Marañón,0.0,86.0,28.0,89.0,99.0
Huallaga,1.0,93.0,19.0,95.0,95.0
Amazonas,1.0,97.0,18.0,91.0,95.0
Cañete-Fortaleza,4.0,95.0,12.0,17.0,97.0
Mantaro,10.0,94.0,19.0,21.0,99.0
Cháparra-Chincha,28.0,99.0,12.0,13.0,82.0
Ucayali,4.0,96.0,8.0,47.0,93.0


Total ΔS of the 14 domains (Gt)


evento,El Niño 2009–10,La Niña 2010–12,El Niño 2015–16,El Niño costero 2017,El Niño costero 2023–24
variante,,,,,
base,-74.2,74.2,-57.6,4.4,63.7
sin_tendencia,-73.7,75.6,-56.3,4.8,64.6


# 3. Sensitivity to the Window Definition (E and C Indices)


In [5]:
display(ev['sensibilidad'].round(2))

,variante,evento,ventana,r,mismo_signo
0,EC_0.5,El Niño 2009–10,2009-08/2010-04,0.99,14.0
1,EC_0.5,La Niña 2010–12,2010-06/2012-06,0.87,14.0
2,EC_0.5,El Niño 2015–16,2014-10/2016-06,0.87,14.0
3,EC_0.5,El Niño costero 2017,2017-02/2017-04,0.98,14.0
4,EC_0.5,El Niño costero 2023–24,2023-03/2024-02,0.98,14.0
5,EC_regimen,El Niño 2009–10,2009-08/2010-04,0.99,14.0
6,EC_regimen,La Niña 2010–12,2010-08/2012-05,0.87,14.0
7,EC_regimen,El Niño 2015–16,2015-06/2016-03,0.88,12.0
8,EC_regimen,El Niño costero 2017,no detectado,NaN,NaN
9,EC_regimen,El Niño costero 2023–24,2023-03/2024-01,0.98,14.0


### 📊 Reading the Results

The coastal events of 2017 and 2023–2024 increase storage in the north, whereas El Niño 2009–2010 and 2015–2016
reduce it in the northern Andes–Amazon and the rainforest, and La Niña 2010–2012 increases it. In 2017, outside the
El Niño E regime, the north gains and the southern highlands lose storage, with changes in the tails of their
distribution that do not vary when the trend is removed. The analysis is descriptive: the events were chosen for
their known impact and the percentiles are not hypothesis tests (thesis, Section 4.4.4).
